# 02 · Vectorless: navegar el documento en vez de trocearlo

**Idea:** en contratos, manuales y normas, la respuesta vive en una **cláusula completa** que el troceado fijo suele partir.
Vectorless (o «RAG por árbol») no usa embeddings: indexa la **estructura** (capítulos → cláusulas, con título y resumen) y
**navega de arriba hacia abajo** hasta la sección correcta.

En este notebook vas a comparar, sobre un contrato de 10 cláusulas:
- **Trozos fijos + vectorial** (40 y 120 palabras, top-3).
- **Árbol sin vectores**: elegir capítulo y luego cláusula usando título + resumen.

⚠️ Aquí el «navegador» es BM25 sobre títulos y resúmenes, para que corra sin API key. En producción quien razona sobre el árbol suele ser un LLM.

## Glosario · vectorless

Lo que necesitas para leer este notebook (puedes volver aquí cuando una palabra no te suene):

- **Embedding:** lista de números que representa el *significado* de un texto; textos con sentido parecido quedan «cerca», aunque no compartan palabras.
- **Similitud coseno:** qué tan cerca están dos embeddings (cerca de 1 = parecidos; cerca de 0 = nada que ver).
- **BM25:** puntaje por *palabras exactas*: cuenta qué documentos contienen las palabras de la pregunta y valora más las raras (`ERR-5012`) que las comunes. No entiende sinónimos; si no comparte ninguna palabra, no devuelve nada. Su puntaje no tiene tope y no es comparable con el coseno.
- **Top-k:** los *k* primeros resultados del ranking (top-3 = los tres mejores).
- **Trocear / ventana de N palabras (*chunk*):** partir el documento en trozos de N palabras (con solape) antes de indexarlo.
- **Vectorless (sin vectores):** en vez de trocear y comparar embeddings, se *navega* la estructura del documento (capítulo → cláusula).
- **Árbol del documento:** cada nodo tiene título y resumen; el buscador baja de capítulo a cláusula.
- **LLM:** modelo de lenguaje (el que redacta o razona). Aquí lo sustituye un BM25 sobre títulos y resúmenes: por eso es un *navegador léxico*.
- **Hecho en contexto / cláusula íntegra:** si el dato necesario aparece en lo recuperado / si llega la cláusula completa, sin cortarla.

In [1]:
import importlib.util, subprocess, sys

for paquete, modulo in [("rank-bm25", "rank_bm25"), ("sentence-transformers", "sentence_transformers")]:
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", paquete])
print("Entorno listo")

Entorno listo


In [2]:
#@title Contrato de transporte (documento largo con estructura) { display-mode: "form" }
# (capitulo, clausula, titulo, resumen, texto). El resumen lo escribiría un LLM al ingestar el documento.
CONTRATO = [
    ("I. Operación", "1", "Objeto", "Alcance del servicio de transporte y qué incluye.",
     "El Proveedor se obliga a transportar la mercadería que Andina Retail le entregue en el almacén central de Lurín hacia las tiendas de Lima, Arequipa y Cusco. El servicio incluye la carga, el traslado y la descarga en el punto de entrega indicado por la Empresa. No incluye el almacenamiento prolongado ni el armado de productos. Cada despacho se solicita mediante una orden de transporte emitida por la Empresa."),
    ("I. Operación", "2", "Plazos de entrega", "Horas máximas de entrega según destino.",
     "La entrega en Lima Metropolitana se realiza dentro de 48 horas desde la recepción de la orden de transporte. Para Arequipa y Cusco el plazo máximo es de 96 horas. Los plazos se cuentan en horas corridas e incluyen sábados, domingos y feriados. Si el Proveedor prevé un retraso, debe avisar a la Empresa de inmediato y proponer una nueva fecha."),
    ("I. Operación", "3", "Penalidades", "Penalidad por retraso y su tope.",
     "Por cada día de retraso en la entrega, el Proveedor pagará una penalidad equivalente al 2% del valor del envío afectado. La penalidad acumulada no podrá exceder el 10% del valor del envío. Las penalidades se descontarán de la siguiente factura. No se aplican penalidades cuando el retraso se deba a caso fortuito o a fuerza mayor debidamente sustentada."),
    ("II. Condiciones económicas", "4", "Tarifas y reajuste", "Cómo se calcula la tarifa y cuánto puede subir cada año.",
     "La tarifa se calcula por kilogramo transportado según la zona de destino y consta en el anexo de tarifas. Cada doce meses la tarifa se reajusta según la variación del índice de precios al consumidor, con un tope máximo de 4%. Cualquier otro incremento requiere acuerdo escrito de ambas partes."),
    ("II. Condiciones económicas", "5", "Facturación y pago", "Emisión de facturas y plazo de pago.",
     "El Proveedor emitirá una factura mensual por los servicios prestados, adjuntando las guías de remisión firmadas. La Empresa pagará dentro de los 45 días calendario siguientes a la recepción conforme de la factura. Los pagos fuera de plazo generan interés moratorio a la tasa legal."),
    ("II. Condiciones económicas", "6", "Seguros", "Cobertura del seguro de transporte y deducible.",
     "El Proveedor mantendrá un seguro de transporte que cubra pérdida, robo o daño de la mercadería hasta el 100% del valor declarado en la orden de transporte. El deducible a cargo del Proveedor es de 1% del valor del siniestro. El certificado de la póliza se entregará a la Empresa antes del primer despacho."),
    ("III. Aspectos legales", "7", "Confidencialidad", "Reserva de la información comercial y su duración.",
     "Las partes mantendrán en reserva la información comercial, los volúmenes de venta y las tarifas a las que accedan por este contrato. La obligación de confidencialidad se mantiene durante cinco años después de terminado el contrato. Solo se puede revelar información por mandato de autoridad competente."),
    ("III. Aspectos legales", "8", "Terminación", "Cómo y cuándo puede terminar el contrato.",
     "Cualquiera de las partes puede terminar el contrato sin expresión de causa con un aviso escrito de 60 días. La Empresa puede resolverlo de inmediato ante incumplimientos graves, como la pérdida reiterada de mercadería o la falta de seguro vigente. Al terminar, el Proveedor entregará toda la documentación pendiente."),
    ("III. Aspectos legales", "9", "Subcontratación", "Condiciones para subcontratar el servicio.",
     "El Proveedor no podrá subcontratar total ni parcialmente el servicio sin autorización escrita previa de la Empresa. El Proveedor sigue siendo responsable frente a la Empresa por los actos de sus subcontratistas, incluidos los retrasos y los daños."),
    ("III. Aspectos legales", "10", "Disposiciones generales", "Notificaciones, ley aplicable y solución de controversias.",
     "Las comunicaciones entre las partes se enviarán a los correos electrónicos designados en el anexo de contactos. Cualquier controversia se someterá a arbitraje de derecho ante la Cámara de Comercio de Lima. El contrato se rige por las leyes de la República del Perú."),
]

# (pregunta, hecho que debe aparecer en el contexto, tipo de vocabulario, cláusula que contiene la respuesta)
PREGUNTAS = [
    ("¿Cuántas horas tiene el proveedor para entregar en Cusco?", "96 horas", "léxica", "2"),
    ("¿Con cuántos días de aviso se puede terminar el contrato?", "60 días", "léxica", "8"),
    ("¿En cuántos días se paga una factura?", "45 días", "léxica", "5"),
    ("¿Cuánto es el deducible del seguro?", "es de 1%", "léxica", "6"),
    ("¿Cuál es el máximo que puede subir la tarifa por año?", "tope máximo de 4%", "léxica", "4"),
    ("¿Durante cuánto tiempo hay que guardar reserva de la información?", "cinco años", "léxica", "7"),
    ("¿Cuál es el tope de las multas por entregar tarde?", "10% del valor", "sinónimo", "3"),
    ("¿Puede el transportista pasar el trabajo a otra empresa?", "autorización escrita previa", "sinónimo", "9"),
    ("¿Dónde se resuelven los conflictos entre las partes?", "Cámara de Comercio de Lima", "sinónimo", "10"),
    ("¿Qué pasa con la penalidad si hay una huelga que impide entregar?", "fuerza mayor", "sinónimo", "3"),
]
print(len(CONTRATO), "cláusulas ·", len(PREGUNTAS), "preguntas")

10 cláusulas · 10 preguntas


In [3]:
import re, numpy as np, pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

STOP = set("a al con cada como de del el en es la las lo los me mi o para por que qué se si su sus un una y cuál cuántos cuántas".split())
tok = lambda s: [w for w in re.findall(r"[\wáéíóúñ]+", s.lower()) if w not in STOP]
palabras = lambda s: len(s.split())
modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

## 1. Troceado fijo + vectorial (el estándar)

Se parte el contrato en ventanas de N palabras (con solape del 25 %), se vectoriza y se traen los 3 trozos más parecidos.

In [4]:
def trocear(n):
    texto = " ".join(f"{t}. {x}" for _, _, t, _, x in CONTRATO).split()
    paso = max(1, int(n * 0.75))
    return [" ".join(texto[i:i + n]) for i in range(0, len(texto), paso)]


def recuperar_trozos(q, trozos, emb, k=3):
    s = emb @ modelo.encode([q], normalize_embeddings=True)[0]
    return " ".join(trozos[i] for i in np.argsort(-s)[:k])


indices = {}
for n in (40, 120):
    tr = trocear(n)
    indices[n] = (tr, modelo.encode(tr, normalize_embeddings=True))
    print(f"ventana de {n} palabras → {len(tr)} trozos")

ventana de 40 palabras → 18 trozos
ventana de 120 palabras → 6 trozos


## 2. Árbol: capítulo → cláusula, sin vectores

Cada nodo tiene `título + resumen` (lo que un LLM generaría al ingestar). Se puntúan los capítulos, se toman los 2 mejores
(*beam*) y dentro de ellos se elige la cláusula; se devuelve la cláusula **entera**.

In [5]:
capitulos = {}
for cap, num, titulo, resumen, texto in CONTRATO:
    capitulos.setdefault(cap, []).append((num, titulo, resumen, texto))

# Un capítulo se describe con los títulos y resúmenes de sus cláusulas
bm_cap = BM25Okapi([tok(cap + " " + " ".join(f"{t} {r}" for _, t, r, _ in cl)) for cap, cl in capitulos.items()])
nombres_cap = list(capitulos)


def recuperar_arbol(q, beam=2):
    sc = bm_cap.get_scores(tok(q))
    candidatas = []
    for i in np.argsort(-sc)[:beam]:
        for num, titulo, resumen, texto in capitulos[nombres_cap[i]]:
            candidatas.append((num, titulo, resumen, texto))
    bm = BM25Okapi([tok(f"{t} {r}") for _, t, r, _ in candidatas])
    mejor = candidatas[int(np.argmax(bm.get_scores(tok(q))))]
    return f"Cláusula {mejor[0]} ({mejor[1]}). {mejor[3]}", mejor[0]


ctx, clausula = recuperar_arbol("¿Con cuántos días de aviso se puede terminar el contrato?")
print("Navegó hasta la cláusula", clausula, "\n", ctx)

Navegó hasta la cláusula 8 
 Cláusula 8 (Terminación). Cualquiera de las partes puede terminar el contrato sin expresión de causa con un aviso escrito de 60 días. La Empresa puede resolverlo de inmediato ante incumplimientos graves, como la pérdida reiterada de mercadería o la falta de seguro vigente. Al terminar, el Proveedor entregará toda la documentación pendiente.


## 3. Comparación

Medimos tres cosas por método:
- **hecho**: ¿el dato necesario aparece en el contexto recuperado?
- **cláusula íntegra**: ¿llega la cláusula completa (lo que necesitas para citarla sin cortar a media frase)?
- **palabras**: cuánto contexto le pasas al LLM (costo y ruido).

Las preguntas «léxicas» comparten palabras con el título o el resumen; las de «sinónimo» no (multas ↔ penalidad, conflictos ↔ controversias).

In [6]:
filas = []
texto_clausula = {num: texto for _, num, _, _, texto in CONTRATO}
for q, hecho, tipo, clausula in PREGUNTAS:
    for n in (40, 120):
        ctx = recuperar_trozos(q, *indices[n])
        filas.append((f"trozos {n} + vectorial", tipo, hecho.lower() in ctx.lower(), texto_clausula[clausula] in ctx, palabras(ctx)))
    ctx, _ = recuperar_arbol(q)
    filas.append(("árbol sin vectores", tipo, hecho.lower() in ctx.lower(), texto_clausula[clausula] in ctx, palabras(ctx)))

df = pd.DataFrame(filas, columns=["método", "tipo", "hecho", "cláusula íntegra", "palabras"])
resumen = df.groupby("método")[["hecho", "cláusula íntegra", "palabras"]].mean().round(2)
por_tipo = df.pivot_table(index="método", columns="tipo", values="hecho", aggfunc="mean").round(2)
print("Promedios globales"); display(resumen)
print("Acierto del hecho según el tipo de vocabulario"); display(por_tipo)

Promedios globales


,hecho,cláusula íntegra,palabras
método,,,
trozos 120 + vectorial,1.0,0.7,348.6
trozos 40 + vectorial,0.9,0.0,120.0
árbol sin vectores,0.7,0.7,59.9


Acierto del hecho según el tipo de vocabulario


tipo,léxica,sinónimo
método,,
trozos 120 + vectorial,1.00,1.00
trozos 40 + vectorial,1.00,0.75
árbol sin vectores,0.83,0.50


### Lectura honesta

- En un contrato **de 10 cláusulas** el troceado vectorial ya funciona bien: aquí no hay ventaja de precisión para el árbol.
- El árbol sí manda **mucho menos contexto** y entrega la **cláusula íntegra** (útil para citar), algo que los trozos chicos casi nunca logran.
- Con un navegador de palabras clave el árbol falla cuando la pregunta no comparte vocabulario (incluso «factura» ≠ «facturación», porque no hay lematización).
  Un LLM que lea el índice lo resolvería, a cambio de latencia y costo por pregunta: ese es el trade-off real de la familia.
- No extrapoles: para saber si vectorless paga en tus documentos (100+ páginas, jerarquía profunda) tienes que medirlo ahí.

In [7]:
# Afirmaciones de la lectura, verificadas con los datos de esta ejecución
assert resumen.loc["árbol sin vectores", "palabras"] < resumen.loc["trozos 120 + vectorial", "palabras"] / 2, "el árbol debería usar menos de la mitad del contexto de trozos de 120"
assert resumen.loc["árbol sin vectores", "cláusula íntegra"] > resumen.loc["trozos 40 + vectorial", "cláusula íntegra"], "el árbol debería entregar más cláusulas íntegras que trozos de 40"
assert por_tipo.loc["árbol sin vectores", "sinónimo"] < por_tipo.loc["árbol sin vectores", "léxica"], "con sinónimos el árbol sin LLM debería caer"
assert resumen.loc["trozos 120 + vectorial", "hecho"] >= resumen.loc["árbol sin vectores", "hecho"], "en este documento corto los trozos grandes deberían acertar al menos igual"
print("Las afirmaciones de la lectura se cumplen con estos datos")

Las afirmaciones de la lectura se cumplen con estos datos


## Qué te llevas

| Usa vectorless cuando… | Evítalo cuando… |
|---|---|
| El documento es largo y **tiene estructura** (contratos, normas, manuales, informes). | Tienes miles de documentos sueltos sin jerarquía. |
| La respuesta es una **sección completa** y debe ser citable. | La pregunta usa lenguaje muy distinto al del índice y no hay LLM que lo interprete. |
| Quieres **trazabilidad**: «la respuesta sale de la cláusula 8». | El presupuesto de latencia es muy bajo (navegar con LLM suma llamadas). |

➡️ Siguiente: `03_graphrag_multihop.ipynb` — preguntas que necesitan **unir** varios documentos.

---
*Webinar «RAG para Soluciones Productivas» · Manuel Argüelles · datos 100 % sintéticos de «Andina Retail»: las cifras ilustran un criterio de decisión, no miden tu sistema. Mide siempre con tus propios datos.*